# Setwise distillation pilot

The teacher compares **k documents at once** and names a winner. Those choices are the training labels. The student scores **one document at a time**, trained with a setwise loss, and at inference the list is sorted by that score.

This run is a small pilot on Jusbrasil:

- **Train:** 50 queries, all of their candidate documents. The teacher labels these.
- **Validation:** 50 other queries, used only to measure nDCG. The teacher is not called.
- **Test:** every query in `small_test.parquet`.

Human `relevance` is only for nDCG. The training target is the teacher winner.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import DistillConfig
from src.data.schema import load_examples, save_examples
from src.models.teacher import SetwiseTeacher

## Data

`train.parquet` and `vali.parquet` are cut down to 50 queries each. Every candidate for a chosen query stays in the frame. `small_test.parquet` is used whole.

In [2]:
cfg = DistillConfig(
    dataset_name="jusbrasil",
    teacher_client="ollama",
    teacher_model="llama3.1:8b",
    k_size=4,
    sets_per_query=2,
    max_doc_chars=1500,
)

DATA_DIR = Path("/Users/david/Documents/data/jusbrasil/manual/with_body/splits")
N_PILOT_QUERIES = 50


def take_queries(df, n_queries, seed):
    """Keep every document from n_queries distinct queries."""
    queries = pd.Series(df["query"].unique())
    if len(queries) <= n_queries:
        return df.copy()
    chosen = set(queries.sample(n=n_queries, random_state=seed))
    return df[df["query"].isin(chosen)].copy()


train_all = pd.read_parquet(DATA_DIR / "train.parquet")
val_all = pd.read_parquet(DATA_DIR / "vali.parquet")
test_df = pd.read_parquet(DATA_DIR / "small_test.parquet")

# train_df = take_queries(train_all, N_PILOT_QUERIES, cfg.seed)
# val_df = take_queries(val_all, N_PILOT_QUERIES, cfg.seed)


train_df = train_all
val_df = val_all  


def describe(name, frame):
    sizes = frame.groupby("query").size()
    ready = int((sizes >= cfg.k_size).sum())
    print(
        f"{name}: {frame['query'].nunique()} queries, {len(frame)} rows, "
        f"{ready} queries with at least {cfg.k_size} docs"
    )


describe("train", train_df)
describe("val", val_df)
describe("test", test_df)

train: 596 queries, 10015 rows, 593 queries with at least 4 docs
val: 109 queries, 1918 rows, 109 queries with at least 4 docs
test: 30 queries, 510 rows, 30 queries with at least 4 docs


## 1. Teacher

One comparison on the first pilot query, before labeling all 50. The reply should be an index into the set, or `-1` when the letter cannot be parsed.

In [3]:
teacher = SetwiseTeacher(
    api_client_type=cfg.teacher_client,
    model_name=cfg.teacher_model,
    k_size=cfg.k_size,
    dataset_name=cfg.dataset_name,
    max_chars=cfg.max_doc_chars,
)

sample = train_df.groupby("query").head(cfg.k_size)
query_text = sample["query"].iloc[0]
docs = sample.to_dict(orient="records")
print(teacher.format_doc(docs[0])[:400])
print("winner index:", teacher.compare_documents_setwise(docs, query_text))

Title: AÇÃO RESCISÓRIA 8064350920224050000
Court: TRF_5
Text: PROCESSO Nº: 0806435-09.2022.4.05.0000 - AÇÃO RESCISÓRIA AUTOR: MUNICIPIO DE BELEM DE MARIA ADVOGADO: Luis Alberto Gallindo Martins RÉU: UNIÃO FEDERAL RELATOR  A : Desembargador  a  Federal Rogério de Meneses Fialho Moreira - 1ª Seção MAGISTRADO CONVOCADO: Desembargador  a  Federal Rafael Chalegre Do Rego Barros EMENTA AGRAVO INTERNO. A
Error comparing documents: Connection error.
winner index: -1


## 2. Labels

`from_samples` draws `sets_per_query` subsets of size `k` from each of the 50 training queries and asks the teacher which document wins. Queries with fewer than `k` documents are skipped.

`from_rank_trace` is the alternative: run the heapsort and keep every comparison it made.

In [4]:
from src.engine.generator import SetwiseLabelGenerator

generator = SetwiseLabelGenerator(
    teacher,
    sets_per_query=cfg.sets_per_query,
    seed=cfg.seed,
)
generated = generator.from_samples(train_df)
print(len(generated.examples), "examples,", generated.skipped, "skipped")
print(generated.queries_seen, "train queries seen")
generated.examples[0]

/Users/david/Library/Caches/pypoetry/virtualenvs/llm-as-judge-for-ltr-xT0j8fzj-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sampling setwise labels: 100%|██████████| 596/596 [2:06:42<00:00, 12.76s/it]  
2026-09-30 15:34:56,218 INFO src.engine.generator: sampled labels: 1186 examples, 0 skipped, 596 queries


1186 examples, 0 skipped
596 train queries seen


SetwiseExample(query='ação anulatória e ação rescisória', documents=['Title: Apelação Cível 1279684620158060001 Fortaleza\nCourt: TJ_CE\nText: GABINETE DESEMBARGADORA CLEIDE ALVES DE AGUIAR NÚMERO ÚNICO: 0127968-46.2015.8.06.0001 TIPO DO PROCESSO: APELAÇÃO CÍVEL EM AÇÃO ANULATÓRIA CUMULADA COM PEDIDO DE ANTECIPAÇÃO DOS EFEITOS DA TUTELA ORIGEM: 23a VARA CÍVEL DA COMARCA DE FORTALEZA APELANTE: FORTALEZA ESPORTE CLUBE APELADOS: GIOVANNI FERNANDES SANTOS E LUIZ CESAR SILVA CAETANO ÓRGÃO JULGADOR: TERCEIRA CÂMARA DE DIREITO PRIVADO RELATORA: DESEMBARGADORA CLEIDE ALVES DE AGUIAR EMENTA: DIREITO PROCESSUAL CIVIL. APELAÇÃO CÍVEL. AÇÃO ANULATÓRIA CUMULADA COM PEDIDO DE ANTECIPAÇÃO DOS EFEITOS DA TUTELA. ANULAÇÃO DE TERMO DE ACORDO HOMOLOGADO JUDICIALMENTE. SENTENÇA HOMOLOGATÓRIA DE TRANSAÇÃO QUE APENAS FORMALIZOU ATO RESULTANTE DA VONTADE DAS PARTES. APLICAÇÃO DO ART. 486 DO CÓDIGO DE PROCESSO CIVIL DE 1973 . VIGÊNCIA À ÉPOCA. CABIMENTO DE AÇÃO ANULATÓRIA, E NÃO DE AÇÃO RESCISÓRIA. INAPLICABI

In [4]:
LABELS_PATH = ROOT / "notebooks" / "data" / "setwise_labels.jsonl"
if "generated" in globals():
    save_examples(generated.examples, LABELS_PATH)
examples = load_examples(LABELS_PATH)
len(examples)

1186

## 3. Setwise loss

Implement `setwise_loss`. For each set, the student scores are logits and the teacher winner is the class. Average the cross-entropy over the batch.

Do not use this function at inference. Inference sorts by the raw score.

`listmle_loss` and `pairwise_ranknet_loss` are optional. Leave them until the setwise path trains.

In [5]:
import torch

from src.loss.setwise import setwise_loss

scores = torch.tensor([[2.0, 0.0, 0.0]])
winner = torch.tensor([0])
value = float(setwise_loss(scores, winner))
print(value)
assert abs(value - 0.2395) < 1e-3

0.2395448386669159


## 4. Student

Implement a pointwise scorer in `StudentRanker`. `score_sets` is already written: it flattens each set, calls `score_pairs`, and reshapes to `[batch, k]`.

A cross-encoder is the usual model. Load it in `__init__` and return one logit per pair from `score_pairs`. Higher means more relevant. Install Transformers with `poetry add transformers` when you get there.

The trainer raises a clear error if the module has no parameters.

Each epoch prints the training loss and nDCG@10 on the 50 training queries, and appends that row to `notebooks/data/train_log.csv`. After the run:

```bash
poetry run python scripts/plot_learning_curve.py notebooks/data/train_log.csv
```

`SANITY_EPOCHS` is set above 1 so the curve can fall. The loss is the setwise loss. nDCG@10 uses the human grades on the same queries.

In [6]:
from src.engine.trainer import SetwiseLoader, SetwiseTrainer
from src.models.student import StudentRanker

LOG_PATH = ROOT / "notebooks" / "data" / "train_log.csv"
SANITY_EPOCHS = 5

student = StudentRanker(cfg.student_model)
loader = SetwiseLoader(examples, batch_size=cfg.batch_size, shuffle=True, seed=cfg.seed)
trainer = SetwiseTrainer(student, setwise_loss, lr=cfg.learning_rate)
history = trainer.fit(
    loader,
    epochs=SANITY_EPOCHS,
    log_path=LOG_PATH,
    ndcg_frame=train_df,
    format_fn=teacher.format_doc,
)
history

/Users/david/Library/Caches/pypoetry/virtualenvs/llm-as-judge-for-ltr-xT0j8fzj-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 254/254 [00:02<00:00, 93.75it/s] 
[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


LoRA student: 2,296,832 trainable parameters of 3,215,052,800


scoring queries: 100%|██████████| 596/596 [1:42:12<00:00, 10.29s/it]
2026-09-30 21:02:23,473 INFO src.engine.trainer: epoch 1  train_loss nan  train_ndcg@10 0.5173


epoch 1  train_loss nan  train_ndcg@10 0.5173


scoring queries: 100%|██████████| 596/596 [1:42:11<00:00, 10.29s/it]
2026-10-01 01:34:09,379 INFO src.engine.trainer: epoch 2  train_loss nan  train_ndcg@10 0.5173


...

epoch 3  train_loss nan  train_ndcg@10 0.5173


scoring queries: 100%|██████████| 596/596 [1:41:13<00:00, 10.19s/it]
2026-10-01 10:36:47,367 INFO src.engine.trainer: epoch 4  train_loss nan  train_ndcg@10 0.5173


epoch 4  train_loss nan  train_ndcg@10 0.5173


epoch 5 train:  18%|█▊        | 215/1186 [30:49<2:19:13,  8.60s/it, loss=nan]


KeyboardInterrupt: 

## 5. Pointwise inference

Each candidate is scored alone with `teacher.format_doc`, then sorted. nDCG uses the human grade.

Validation is every query in `vali.parquet`. Test is every query in `small_test.parquet`. The teacher is not called. Run this cell only after the training cell prints all five epochs.

In [ ]:
from src.engine.evaluator import evaluate_pointwise

val_metrics = evaluate_pointwise(
    student, val_df, format_fn=teacher.format_doc, batch_size=4
)
test_metrics = evaluate_pointwise(
    student, test_df, format_fn=teacher.format_doc, batch_size=4
)
print("val ", val_metrics.means)
print("test", test_metrics.means)

val  {'ndcg@1': 0.6323809523809524, 'ndcg@3': 0.7020229975235809, 'ndcg@5': 0.7142120152424417, 'ndcg@10': 0.753950056774543}
test {'ndcg@1': 0.7365079365079363, 'ndcg@3': 0.7945559893914135, 'ndcg@5': 0.8078065727464884, 'ndcg@10': 0.8266049882903104}
